In [ ]:
import geopandas as gdp
import numpy as np
import geopandas as gpd
from shapely.geometry import LineString
from scipy.stats import gaussian_kde

from scipy.interpolate import splprep, splev


gdf = gdp.read_file("../../data/processed/trajectory_clusters.gpkg")

In [ ]:
centerlines = []
cluster_ids = []
traj_ids = []
depth_readings = []

for cluster_id in gdf["cluster"].unique():

    if cluster_id == -1:
        continue

    subset = gdf[gdf["cluster"] == cluster_id]
    
    print(subset['traj_id'])

    # convert to points 
    points = []
    for geom in subset.geometry:
        for x, y in geom.coords:
            points.append([x, y])

    points = np.array(points)

    if len(points) < 5:
        print(f"not enough points")
        continue

    kde = gaussian_kde(points.T, bw_method = 0.05)

    xmin, ymin, xmax, ymax = subset.total_bounds
    grid_size = 5

    xgrid = np.arange(xmin, xmax, grid_size)
    ygrid = np.arange(ymin, ymax, grid_size)

    X, Y = np.meshgrid(xgrid, ygrid)
    grid_coords = np.vstack([X.ravel(), Y.ravel()])

    Z = kde(grid_coords).reshape(X.shape)

    # ridge extraction 
    ridge_points = []
    for i in range(Z.shape[0]):
        Z = kde(grid_coords)
        threshold = np.percentile(Z, 10)  # only the top 10% densest areas are kept
        mask = Z > threshold
        ridge_points = grid_coords[:, mask].T

    coords = np.array(ridge_points)

    coords = np.unique(coords, axis=0)

    if len(coords) < 2:
        print(f"invalid geometry")
        continue

    coords = coords[np.argsort(coords[:, 0])]

    if len(coords) < 4:
        line = LineString(coords)
    else:
        try:
            tck, _ = splprep([coords[:, 0], coords[:, 1]], s=10, k=3)
            u_new = np.linspace(0, 1, 200)
            x_smooth, y_smooth = splev(u_new, tck)
            line = LineString(zip(x_smooth, y_smooth))
        except Exception as e:
            print(f"Spline extraction failed for cluster {cluster_id}: {e}")
            line = LineString(coords)
    
    centerlines.append(line)
    cluster_ids.append(cluster_id)
    traj_ids.append(subset['traj_id'].unique())
    depth_readings.append(subset['DRVAL2'].unique())

In [ ]:
gdf_centerlines = gpd.GeoDataFrame(
    {"cluster": cluster_ids, "traj_id": traj_ids, "depth_readings": depth_readings},
    geometry=centerlines,
    crs=gdf.crs
)


In [ ]:
gdf_centerlines['string_traj_ids'] = gdf_centerlines['traj_ids'].apply(lambda x: ', '.join(map(str, x)))


In [ ]:
gdf_centerlines.to_file("../../data/processed/corridor_centerlines_ridge.gpkg", driver="GPKG")